# 실험 03: 출력 파서 (Output Parser)

## 1. 실험 제목과 목표
- **제목**: AI 응답 구조화 및 데이터 타입 변환
- **목표**: 모델이 뱉어내는 원시 형태의 `AIMessage` 객체를 서비스에서 바로 쓸 수 있는 문자열(String)이나 리스트(List) 구조로 자동 변환해주는 파서의 역할을 알아봅니다.

## 2. 실험 개요
1. **비교 실험 1**: 메타데이터가 섞인 `AIMessage` 객체 vs `StrOutputParser`로 정제된 깔끔한 문자열
2. **비교 실험 2**: 모델의 응답을 자동으로 파이썬 배열로 바꿔주는 `CommaSeparatedListOutputParser` 관찰
3. **실패/주의 케이스**: 파서를 붙여도 모델이 형식 지시를 어기면 파싱이 엉망이 되는 현상 (프롬프트 결합의 중요성)

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 비교 실험 1: AIMessage vs StrOutputParser
일반적으로 프론트엔드에 넘길 때는 문자열(String)만 필요합니다.

In [3]:
question = "태양계 행성을 3개만 말해줘."

# 1. 파서 없이 호출 (기본)
raw_response = llm.invoke(question)
print("=== [1. 파서 없이 호출 (AIMessage)] ===")
print("데이터 타입:", type(raw_response))
print("객체 자체 출력:", raw_response)
# 결과는 사용할 수 없는 복잡한 메타데이터(토큰 수, 모델 정보 등)가 포함된 객체입니다.

print("\n=== [2. StrOutputParser 사용] ===")
parser = StrOutputParser()

# 파서에 통과(parse) 시킵니다.
parsed_string = parser.parse(raw_response.content) # 또는 raw_response 객체 자체를 넘길 수도 있습니다(LCEL 사용 시)
print("데이터 타입:", type(parsed_string))
print("정제된 출력:\n", parsed_string)

print("\n-> 결과: StrOutputParser는 복잡한 객체의 껍데기를 벗기고 핵심 `.content`만 추출해줍니다.")

=== [1. 파서 없이 호출 (AIMessage)] ===
데이터 타입: <class 'langchain_core.messages.ai.AIMessage'>
객체 자체 출력: content='태양계의 행성 중 3개는 다음과 같습니다:\n\n1. 지구 (Earth)\n2. 화성 (Mars)\n3. 목성 (Jupiter) \n\n더 궁금한 점이 있으면 말씀해 주세요!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 20, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5a1892700', 'id': 'chatcmpl-DvFqvfEebWLVQXY50wQOZlgMqoUto', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f078b-69a6-7922-bb80-186e55db990f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 20, 'output_tokens': 50, 'total_tokens': 70, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

## 6. 비교 실험 2: 파이썬 배열로 변환 (CommaSeparatedListOutputParser)
사용자가 요구한 여러 개의 아이템을 문자열이 아닌 다루기 쉬운 `List` 타입으로 변환합니다.

In [4]:
list_parser = CommaSeparatedListOutputParser()

# 파서가 스스로 자신의 형식 지시사항 문자열을 생성해줍니다.
# "Your response should be a list of comma separated values, eg: `foo, bar, baz`"
format_instructions = list_parser.get_format_instructions()
print("[파서가 자동 생성한 지시사항]")
print(format_instructions)

# 템플릿에 파서의 지시사항을 끼워넣습니다.
prompt = PromptTemplate(
    template="한국의 인기 음식 5가지를 추천해줘.\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": format_instructions}
)

print("\n[모델 호출 및 파싱 결과]")
# 1. 프롬프트 생성
final_prompt = prompt.format()

# 2. 모델 호출
ai_response = llm.invoke(final_prompt)

# 3. 파싱 (문자열 -> 리스트)
parsed_list = list_parser.parse(ai_response.content)

print("데이터 타입:", type(parsed_list))
print("추출된 리스트:", parsed_list)
print("리스트 첫 번째 요소:", parsed_list[0])

[파서가 자동 생성한 지시사항]
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`

[모델 호출 및 파싱 결과]
데이터 타입: <class 'list'>
추출된 리스트: ['불고기', '비빔밥', '김치찌개', '잡채', '떡볶이']
리스트 첫 번째 요소: 불고기


## 7. 실패/주의 케이스: 모델이 지시사항을 무시했을 때의 파싱 에러
파서를 썼다고 무조건 리스트가 되는 것은 아닙니다. 모델이 쉼표(,) 대신 줄바꿈으로 답을 주면 파싱은 엉망이 됩니다.

In [5]:
print("[모델의 포맷 파괴 시뮬레이션]")
bad_prompt = PromptTemplate(
    template="한국의 인기 음식 5가지를 추천해줘. 숫자 목록(1. 2. 3.)으로 예쁘게 줄바꿈해서 답해.",
    input_variables=[]
)

bad_ai_response = llm.invoke(bad_prompt.format())
print("AI 원본 응답:\n", bad_ai_response.content)

print("\n[쉼표 파서 강제 적용]")
ruined_list = list_parser.parse(bad_ai_response.content)
print("추출된 리스트:", ruined_list)

print("\n-> 결과: 파서는 단순히 쉼표(,)를 기준으로 텍스트를 자를 뿐입니다. 모델이 줄바꿈으로 답을 했기 때문에 전체가 통째로 1개의 리스트 요소로 들어가버렸습니다. 파서를 잘 쓰려면 프롬프트 엔지니어링이 반드시 동반되어야 합니다.")

[모델의 포맷 파괴 시뮬레이션]
AI 원본 응답:
 물론입니다! 한국의 인기 음식 5가지를 추천해드릴게요.

1. 김치찌개  
2. 불고기  
3. 비빔밥  
4. 떡볶이  
5. 순두부찌개  

맛있게 드세요!

[쉼표 파서 강제 적용]
추출된 리스트: ['물론입니다! 한국의 인기 음식 5가지를 추천해드릴게요.', '1. 김치찌개  ', '2. 불고기  ', '3. 비빔밥  ', '4. 떡볶이  ', '5. 순두부찌개  ', '맛있게 드세요!']

-> 결과: 파서는 단순히 쉼표(,)를 기준으로 텍스트를 자를 뿐입니다. 모델이 줄바꿈으로 답을 했기 때문에 전체가 통째로 1개의 리스트 요소로 들어가버렸습니다. 파서를 잘 쓰려면 프롬프트 엔지니어링이 반드시 동반되어야 합니다.


## 8. 결과 해석

1. **데이터 규격화**: LLM 애플리케이션 개발의 핵심은 모델의 창의성을 발휘하면서도, 최종 결과물을 시스템이 다루기 쉬운 규격(JSON, List)으로 묶어내는 것입니다.
2. **자동화된 프롬프트**: LangChain 파서들은 단순히 결과만 잘라주는 것이 아니라, `get_format_instructions()`를 통해 모델을 통제할 프롬프트(지시사항)까지 자동으로 생성해주는 편의성을 제공합니다.
3. **GIGO(Garbage In, Garbage Out)**: 파서는 마법이 아니며, 모델이 포맷 지시사항을 무시하면 파싱 로직도 무너집니다. 따라서 신뢰성 있는 구조적 출력을 위해서는 모델의 성능(gpt-4o 등)이 매우 중요합니다.

## 9. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `AIMessage` 객체를 매번 `res.content`로 파싱하는 귀찮음을 `StrOutputParser`가 해결해줌.
* `CommaSeparatedListOutputParser` 등을 쓰면 파이썬 리스트 구조를 매우 쉽게 얻을 수 있으나, 만약 모델이 말을 안 듣고 다른 형식으로 대답하면 파싱 로직이 붕괴됨.

### 다음 개선 방향
* 지금까지는 `prompt -> model -> parser` 단계를 각각 코드로 실행하여 변수에 담아 넘겼음.
* 이 세 단계를 하나로 압축하고 마법처럼 묶어주는 **LCEL(LangChain Expression Language)** 문법을 배워, 코드를 획기적으로 줄일 차례임.